# Demo Notebook: *On the representations of entities in Auto-Regressive Large Language Models*

This notebook demonstrates the experiments descibed in our papier *On the representations of entities in Auto-Regressive Large Language Models*



## Imports and Utils

In [18]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [19]:
#delete
%cd ~/code/entityrepresentations/

/home/morand/code/entityrepresentations


/home/morand/reimplems/taskVectorsHendel/env/lib/python3.10/site-packages/IPython/core/magics/osm.py:393: UserWarning:

This is now an optional IPython functionality, using bookmarks requires you to install the `pickleshare` library.

/home/morand/reimplems/taskVectorsHendel/env/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning:

This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.



In [20]:
import torch, os, gc, sys
from pathlib import Path
from tqdm import tqdm
# read weights files 

repo_path = Path(os.getcwd())
weights_path = repo_path / "weights"
if weights_path.exists():
    weights = list(weights_path.glob("*.pth"))
    print(f"Found weights in {weights_path}")
    for w in weights: print(" -", w.name)
else:
    print(f"Working in directory {os.getcwd()}, please make sure to use the repo root as wd")
    weights = []


Found weights in /home/morand/code/entityrepresentations/weights
 - TaskVec_phi-1_5_l10_e10.pth


In [21]:
import transformer_lens as tl 
from transformer_lens import HookedTransformer, patching


#our own code
import utils 
from LabelExtractor import eval_model, infer_entities
from processResults import *
import circuitsvis as cv
import plotly.io as pio

# Plotly needs a different renderer for VSCode/Notebooks vs Colab argh
pio.renderers.default = "notebook_connected"
print(f"Using renderer: {pio.renderers.default}")
# Testing that the library works
cv.examples.hello("Fellow AI researcher")


Using renderer: notebook_connected


## Params 
Thanks to the [`transformer_lens`](https://github.com/TransformerLensOrg/TransformerLens/tree/main) library, we can load and use many different llms seemlessly.

In [22]:
# Load a model (eg GPT-2 Small)
model_name = "meta-llama/Meta-Llama-3-8B" # ?
model_name = "gpt2-small" # 117M ok
model_name = "pythia-2.8b"#ok !
model_name = "gpt2-xl" # 1.5B ok
model_name = "mistralai/Mistral-7B-v0.1" # ok JZ a100 8cpus | 
model_name = "gpt2-large" # 774M ok
model_name = "gpt2-medium" # 302M ok
model_name = "phi-2" # 2,5B ok 12cpus nope, gpu 24cpus ok
model_name = "phi-1_5"  # 1.5B ok

with_context = True

# load results
xp_path = pathlib.Path.home() / "experiments_JeanZay"
#get all experiments directories
xp_paths= os.listdir(xp_path / "jobs")
jobs_path = xp_path / "jobs" / xp_paths[0]
#get jobs
results = loadResults(jobs_path)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2252/2252 [00:10<00:00, 222.04it/s]


## Load Model

In [23]:
#check if model variable exists
if not 'model' in locals():
    model = utils.load_model(model_name)
    dim = model.QK.shape[-1]
print(model)
model.eval()
model = model.cuda()

HookedTransformer(
  (embed): Embed()
  (hook_embed): HookPoint()
  (blocks): ModuleList(
    (0-23): 24 x TransformerBlock(
      (ln1): LayerNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2): LayerNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (attn): Attention(
        (hook_k): HookPoint()
        (hook_q): HookPoint()
        (hook_v): HookPoint()
        (hook_z): HookPoint()
        (hook_attn_scores): HookPoint()
        (hook_pattern): HookPoint()
        (hook_result): HookPoint()
        (hook_rot_k): HookPoint()
        (hook_rot_q): HookPoint()
      )
      (mlp): MLP(
        (hook_pre): HookPoint()
        (hook_post): HookPoint()
      )
      (hook_attn_in): HookPoint()
      (hook_q_input): HookPoint()
      (hook_k_input): HookPoint()
      (hook_v_input): HookPoint()
      (hook_mlp_in): HookPoint()
      (hook_attn_out): HookPoint()
      (hook_mlp_out): HookPoint()
    

## Load the data

In [24]:
dataset_name = "CoNLL2003"
train_dataset, test_dataset, val_dataset = utils.load_datasets(dataset_name, max_ent_length=200)

In [25]:
print("train length:", len(train_dataset))
print("dev length:", len(val_dataset))
print("test length:", len(test_dataset))
print("ex sample:") 
item = train_dataset[np.random.randint(len(val_dataset))]
for key in item.keys():
    print(" -", key, ":", item[key])

train length: 22749
dev length: 5695
test length: 5389
ex sample:
 - entity : Republic
 - text : Republic) 6-3 6-4
 - id : 3518


## Test TaskVecs

In [26]:
layer = 10
fileName = get_taskVec(results, model_name, layer=layer, dataset_name=dataset_name, with_context = with_context)

TaskVec = torch.load(fileName, weights_only=True)
print("TaskVec loaded from ", fileName)


found 1 jobs for layer 10 of phi-1_5 with context on CoNLL2003 with method in_context.
found ['TaskVec_phi-1_5_l10_e9.996047430830039.pth'] 
TaskVec loaded from  /home/morand/experiments_JeanZay/jobs/labelextractor.learnlabelextractor/fd06cd6ca8717778db373934ccbe3014574392509bee4c04bfd557833c3b9c70/TaskVec_phi-1_5_l10_e9.996047430830039.pth


# Entity Lens
Now that we have a LLM and a trained task vector $\theta_\ell$ loaded, we can infer entities from any representations.
We showcase here the *Entity Lens*, that generates a mention for each token considered at specifed layers, allowing to visualize to what *entity* the model is *thinking* in its internal representations.

In [27]:
ind = np.random.randint(len(test_dataset))
# ind = 7616
context = test_dataset[ind]["text"]
# context = " _ > Agnes Kant"
# context = "Alan Bean was an American astronaut, born on March 15, 1932 in Wheeler, Texas. He received a Bachelor of Science degree at the University of Texas at Austin in 1955 and was chosen by NASA in 1963. _ > Alan Bean"
# context = ...

words = model.to_str_tokens(context)
print(len(words))
cv.tokens.colored_tokens(words, words)


25


In [28]:
#compute whole cache
print("computing cache ...")
#get whole hidden states
_ , cache = model.run_with_cache(context)
repr = cache[tl.utils.get_act_name("resid_post", layer, "")][0,:,:] # 1 x n_tokens x dim
repr = repr.detach().cuda()
print(repr.shape)
data = [
    {   
        "id": i,
        "text": context,
        "representation": repr[i],
        "tok": words[i],
    }
    for i in range(len(words))
]
infer_entities(model, TaskVec, data, with_context=with_context)

computing cache ...
torch.Size([25, 2048])


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:01<00:00,  1.69it/s]


### Raw display

In [29]:
print(data[0]["text"])
print("Token".center(20), "| Inferred")
print("-"*60)
for it in data:
    print(f"{it['tok'].center(20)} | {it['inferred']}" )

Their other marksmen were Brazilian defender Vampeta and Belgian striker Luc Nilis, his 14th of the season.
       Token         | Inferred
------------------------------------------------------------
   <|endoftext|>     | Season
       Their         | their
        other        | other
        marks        | marks
        men          | their other marksmen
        were         | Brazilian defender Vampeta
      Brazilian      | Brazilian
      defender       | Brazilian defender
          V          | V
        amp          | Vamp
        eta          | Vampeta
         and         | 
       Belgian       | Belgian
       striker       | Striker
         Luc         | Luc
         Nil         | Luc Nilis
         is          | Luc Nilis
         ,           | Brazilian defender Vampeta
         his         | second
         14          | 14
         th          | 14th
         of          | 
         the         | 
       season        | season
         .           | Season


### Circuitvis
smoother visualization

In [30]:
html = cv.tokens.colored_tokens(words, [it['inferred'] for it in data])
html.cdn_src = html.cdn_src.replace("margin: 15px", "margin: 50px")
html

### Old IPywidget code

In [31]:
from IPython.display import display
import ipywidgets as widgets

prompt = train_dataset[np.random.randint(len(train_dataset))]["text"]
# prompt = " _ > Agnes Kant"
# prompt = "Alan Bean was an American astronaut, born on March 15, 1932 in Wheeler, Texas. He received a Bachelor of Science degree at the University of Texas at Austin in 1955 and was chosen by NASA in 1963. _ > Alan Bean"
words = model.to_str_tokens(prompt)
print(words)
print(len(words))

# Callback function to update selected word
def on_button_click(b):
    selected_word_label.value = f"Selected Word: {b.description}"
    layer = dropdown.value
    repr = utils.get_representation(model,
                                layer=layer,
                                tokens=model.to_tokens(prompt),
                                token_inds=torch.tensor([b.id]),
                                verbose=True)
    #change button color
    b.style.button_color = 'lightgreen'
    #change all others to default
    for button in buttons:
        if button.id != b.id:
            button.style.button_color = 'white'
    data = [{
        "id":0,
        "representation": repr,
        "text": prompt
        }]
    infer_entities(model, TaskVec, data, with_context=with_context)
    print("generation:", data[0]["inferred"])

# Buttons for each word
buttons = []
class clickableToken(widgets.Button):
    def __init__(self, id:int, description:str, **kwargs):
        super().__init__(description=description, **kwargs)
        self.description = description
        self.id = id
        self.on_click(on_button_click)

for ind, token in enumerate(words):
    buttons.append(
        clickableToken(
            id=ind, 
            description=token,
            layout=widgets.Layout(width='auto', margin='2px', padding='0 5px')
        ))
    

# Box widget with flexible wrapping
button_box = widgets.Box(
    children=buttons,
    layout=widgets.Layout(display='flex', flex_flow='row wrap', align_items='center')
)
# Dropdown widget for selecting an integer
dropdown = widgets.Dropdown(
    options=[(f"layer {i}", i) for i in range(len(model.blocks)-1, -1, -1)],
    description='Select a layer:',
    disabled=False,    
)
dropdown.value = layer
def on_checkbox_change(change):
    global with_context
    with_context = change['new']

# Create the checkbox widget
with_context_checkbox = widgets.Checkbox(
    value=with_context,
    description='Generate with context',
    disabled=False
)
# Link the function to the checkbox change event
with_context_checkbox.observe(on_checkbox_change, names='value')

# Label to display the selected word
selected_word_label = widgets.Label()

# # clean previous display
# display.clear_output() #works ?? 

# Create a title using HTML widget
title = widgets.HTML(value="<h3>Select a Token to generate from:</h3>")

# Display widgets
display(title)
display(button_box)
#display checkbox  and dropdown side by side
display(widgets.HBox([with_context_checkbox, dropdown]))
display(selected_word_label)

['<|endoftext|>', 'Results', ' of', ' second', ' round', ' matches', ' on', ' Thursday', ' in', ' the', ' U', '.', 'S', '.', ' Open', ' Tennis', ' Championships', ' at', ' the', ' National', ' Tennis', ' Centre', ' (', 'prefix', ' number', ' denotes', ' se', 'eding', '):']
29


HTML(value='<h3>Select a Token to generate from:</h3>')

Box(children=(clickableToken(description='<|endoftext|>', layout=Layout(margin='2px', padding='0 5px', width='…

Label(value='')